Cell 1 - Startup / clone / pull

In [1]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/amcguire985/humanoid-internship-challenge.git"
REPO_DIR = Path("/content/humanoid-internship-challenge")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )

os.chdir(REPO_DIR)

subprocess.run(["git", "pull"], check=True)

print("Working directory:", os.getcwd())
print("Latest commit:")
subprocess.run(["git", "log", "-1", "--oneline"])

Working directory: /content/humanoid-internship-challenge
Latest commit:


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

Cell 2 - GPU / SmolVLA dependencies

In [2]:
import sys
import torch
import subprocess
from pathlib import Path

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

requirements = Path("config/smolvla/requirements-gpu.txt")

if requirements.exists():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(requirements)
        ],
        check=True
    )

print("SmolVLA dependencies ready.")

Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
SmolVLA dependencies ready.


Cell 3 - Set up LIBERO environment

In [3]:
import os
import subprocess
from pathlib import Path

MAMBA_ROOT = "/content/micromamba"
LIBERO_ENV = Path(MAMBA_ROOT) / "envs/libero"
LIBERO_PYTHON = LIBERO_ENV / "bin/python"
LIBERO_REPO = Path("/content/LIBERO")

# Install micromamba
if not Path("/usr/local/bin/micromamba").exists():
    subprocess.run(
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest "
        "| tar -xvj bin/micromamba",
        shell=True,
        check=True,
        cwd="/content"
    )
    subprocess.run(
        ["cp", "/content/bin/micromamba", "/usr/local/bin/micromamba"],
        check=True
    )

# Create Python 3.8 environment
if not LIBERO_ENV.exists():
    subprocess.run(
        [
            "micromamba", "create",
            "-y",
            "-r", MAMBA_ROOT,
            "-n", "libero",
            "python=3.8.13",
            "pip"
        ],
        check=True
    )

# Clone LIBERO
if not LIBERO_REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/Lifelong-Robot-Learning/LIBERO.git",
            str(LIBERO_REPO)
        ],
        check=True
    )

# Install LIBERO
subprocess.run(
    [str(LIBERO_PYTHON), "-m", "pip", "install", "-q",
     "-r", str(LIBERO_REPO / "requirements.txt")],
    check=True
)

subprocess.run(
    [str(LIBERO_PYTHON), "-m", "pip", "install", "-q",
     "-e", str(LIBERO_REPO)],
    check=True
)

# Headless rendering libraries
subprocess.run(["apt-get", "update", "-qq"], check=True)

subprocess.run(
    [
        "apt-get", "install", "-y", "-qq",
        "libosmesa6",
        "libosmesa6-dev",
        "libgl1",
        "libglx-mesa0",
        "libegl1",
        "libgles2"
    ],
    check=True
)

print("LIBERO Python:", LIBERO_PYTHON)
print("LIBERO setup complete.")

LIBERO Python: /content/micromamba/envs/libero/bin/python
LIBERO setup complete.


Cell 4 - LIBERO first-run config + import test

In [4]:
import os
import subprocess

LIBERO_PYTHON = "/content/micromamba/envs/libero/bin/python"

env = os.environ.copy()
env["MUJOCO_GL"] = "osmesa"

# LIBERO may ask for its dataset path the first time it imports.
first_run = subprocess.run(
    [
        LIBERO_PYTHON,
        "-c",
        "from libero.libero import benchmark"
    ],
    input="N\n",
    text=True,
    env=env,
    capture_output=True
)

# Actual verification
result = subprocess.run(
    [
        LIBERO_PYTHON,
        "-c",
        (
            "from libero.libero import benchmark; "
            "print('LIBERO import works'); "
            "print(list(benchmark.get_benchmark_dict().keys()))"
        )
    ],
    env=env,
    text=True,
    capture_output=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("LIBERO import failed")

LIBERO import works
['libero_spatial', 'libero_object', 'libero_goal', 'libero_90', 'libero_10', 'libero_100']



Cell 5 - Simulator preflight

In [5]:
import os
import subprocess
from textwrap import dedent

LIBERO_PYTHON = "/content/micromamba/envs/libero/bin/python"

code = dedent("""
import os
os.environ["MUJOCO_GL"] = "osmesa"

from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv
from libero.libero import get_libero_path

suite = benchmark.get_benchmark_dict()["libero_spatial"]()
task = suite.get_task(0)

bddl = os.path.join(
    get_libero_path("bddl_files"),
    task.problem_folder,
    task.bddl_file
)

env = OffScreenRenderEnv(
    bddl_file_name=bddl,
    camera_heights=128,
    camera_widths=128
)

obs = env.reset()

print("Task:", task.language)
print("EEF:", obs["robot0_eef_pos"])
print("Image:", obs["agentview_image"].shape)

env.close()

print("LIBERO preflight passed.")
""")

env = os.environ.copy()
env["MUJOCO_GL"] = "osmesa"

result = subprocess.run(
    [LIBERO_PYTHON, "-c", code],
    env=env,
    text=True,
    capture_output=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[Warning]: datasets path /content/LIBERO/libero/libero/../datasets does not exist!
Task: pick up the black bowl between the plate and the ramekin and place it on the plate
EEF: [-0.20846466  0.          1.17327948]
Image: (128, 128, 3)
LIBERO preflight passed.



Cell 6 - Pull latest changes

In [6]:
import os
import subprocess

os.chdir("/content/humanoid-internship-challenge")

subprocess.run(["git", "pull"], check=True)

print("\nNow at:")
subprocess.run(["git", "log", "-1", "--oneline"])


Now at:


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

Cell 7 - Run trained simulation

Cell 8.0 - Google drive mount

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%bash
set -euo pipefail
cd /content/humanoid-internship-challenge
mountpoint -q /content/drive || { echo "Run cell 8.0 (Drive mount) first."; exit 1; }
export MPLBACKEND=Agg MUJOCO_GL=egl
export LIBERO_CONFIG_PATH=/content/libero-official-config
export OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 LP_NUM_THREADS=2
unset PYTHONPATH
command -v ffmpeg >/dev/null || { apt-get update -qq; apt-get install -y -qq ffmpeg; }
BASELINE=/content/drive/MyDrive/humanoid_results/libero_official_bowl_20261007_204110
for FILE in packages.txt sanity_manifest.json libero_spatial_2/eval_info.json; do
  [ -f "$BASELINE/$FILE" ] || { echo "Missing original baseline: $BASELINE/$FILE"; exit 1; }
done
for FILE in scripts/bowl_synthetic_reference.py config/smolvla/official_sanity_setup.bash config/smolvla/official_sanity_download.bash; do
  [ -f "$FILE" ] || { echo "Missing updated source: $FILE. Sync this checkout before running."; exit 1; }
done
# Cells 1-6 prepare a separate LIBERO-only environment. Keep it intact.
# Prepare the pinned evaluator only on a fresh runtime; neither step runs a rollout.
if [ ! -x /content/libero-official-env/bin/python ]; then
  bash config/smolvla/official_sanity_setup.bash
fi
if [ ! -f /content/smolvla-libero-official-checkpoint/model.safetensors ]; then
  bash config/smolvla/official_sanity_download.bash
fi
/content/libero-official-env/bin/python - <<'PY'
import torch
from lerobot.scripts import lerobot_eval
assert torch.cuda.is_available(), "Select a Colab GPU runtime."
from pathlib import Path
import sys
sys.path.insert(0, 'scripts')
from bowl_synthetic_reference import generate
assert "'D1'" in Path('scripts/evaluate_bowl_hybrid.py').read_text(), "Sync the updated D1 runner."
print('Pinned evaluator available; baseline/checkpoint checks run before simulation.')
PY
RUN_ROOT="$(mktemp -d /content/drive/MyDrive/humanoid_results/bowl_hybrid_synthetic_XXXXXXXX)"
printf '%s\n' "$RUN_ROOT/D1" > /content/bowl_synthetic_last_output.txt
printf '\nResult folder: %s\n' "$(basename "$RUN_ROOT")"
printf 'Google Drive: MyDrive/humanoid_results/%s/D1\n' "$(basename "$RUN_ROOT")"
printf 'Full output path: %s\n\n' "$RUN_ROOT/D1"
/content/libero-official-env/bin/python scripts/evaluate_bowl_hybrid.py \
  --hybrid-condition D1 --synthetic-duration 10.8 \
  --guidance-config config/smolvla/bowl_hybrid.json \
  --placement-config config/smolvla/bowl_placement.json \
  --baseline-root "$BASELINE" \
  --policy.path=/content/smolvla-libero-official-checkpoint \
  --policy.device=cuda --policy.load_vlm_weights=false \
  --env.type=libero --env.task=libero_spatial --env.task_ids="[2]" \
  --env.control_mode=relative --env.init_states=true --env.hard_reset=true \
  --env.observation_height=256 --env.observation_width=256 \
  --env.max_parallel_tasks=1 --eval.batch_size=1 --eval.n_episodes=1 \
  --seed=0 --output_dir="$RUN_ROOT/D1" 2>&1 | tee "$RUN_ROOT/D1.log"
printf '\nSimulation finished. Result folder: %s\n' "$(basename "$RUN_ROOT")"
printf 'Google Drive: MyDrive/humanoid_results/%s/D1\n' "$(basename "$RUN_ROOT")"
printf 'Full output path: %s\n' "$RUN_ROOT/D1"


Cell 9 - Export results

In [ ]:
from pathlib import Path
import json

# Read the exact D1 run from cell 8; do not rerun or reprocess historical C.
pointer = Path("/content/bowl_synthetic_last_output.txt")
if not pointer.exists():
    raise RuntimeError("Run cell 8 first, or set condition to its printed D1 path.")
condition = Path(pointer.read_text().strip())
summary = condition / "summary.json"
print("D1 output:", condition)
if not summary.exists():
    raise RuntimeError(f"D1 did not finish reporting. Inspect {condition.parent / 'D1.log'}; do not automatically rerun.")
print(json.dumps(json.loads(summary.read_text()), indent=2))
print("Plots, reference and videos:", condition / "ground_truth/episode_000")


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/humanoid_results")

for run in sorted(root.glob("bowl_hybrid*")):
    print(f"\nRUN: {run}")
    for name in ("summary.json", "eval_info.json", "trajectory.jsonl"):
        matches = list(run.rglob(name))
        for path in matches:
            print(f"  {path.relative_to(root)}")
        if not matches:
            print(f"  No {name} found")


RUN: /content/drive/MyDrive/humanoid_results/bowl_hybrid_20261007_225314
  No summary.json found
  No eval_info.json found
  bowl_hybrid_20261007_225314/C/ground_truth/episode_000/trajectory.jsonl

RUN: /content/drive/MyDrive/humanoid_results/bowl_hybrid_20261007_231712
  bowl_hybrid_20261007_231712/C/summary.json
  bowl_hybrid_20261007_231712/C/ground_truth/episode_000/summary.json
  bowl_hybrid_20261007_231712/C/eval_info.json
  bowl_hybrid_20261007_231712/C/ground_truth/episode_000/trajectory.jsonl
